# Comprehensive Model Comparison: Optic Disc/Cup Segmentation

This notebook evaluates and compares five different approaches for optic disc and cup segmentation on the test dataset.

**Approaches:**
1. **Baseline UNet** - Standard UNet architecture
2. **CLAHE** - UNet trained with CLAHE preprocessing
3. **Baseline + Std Enhancer** - UNet with learned standard enhancer
4. **Baseline + Atrous Enhancer** - UNet with learned multi-scale enhancer
5. **ASPP Unet** - UNet with Atrous Spatial Pyramid Pooling

**Metrics:**
- Loss, IoU BG, IoU Disc, IoU Cup, mIoU, Parameters, Δ mIoU

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
import pandas as pd

# Add src to path
project_root = Path.cwd().parent
sys.path.append(str(project_root / 'src'))

from data_loader.dataset import GlaucomaDataset
from data_loader.transforms import get_validation_transforms
from models.unet import UNet
from models.aspp_unet import ASPPUNet
from models.enhancer import ImageEnhancer
from models.atrous_enhancer import AtrousImageEnhancer
from training.train import CombinedLoss, DiceLoss
from training.train_enhancer import EnhancerUNetModel

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Load Test Dataset

In [ ]:
# Load test dataset
test_transforms = get_validation_transforms(image_size=256, use_clahe=False)
test_dataset = GlaucomaDataset(
    root_dir=str(project_root),
    split='test',
    transform=test_transforms,
    seed=42,
    filter_incomplete=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print(f"Test dataset: {len(test_dataset)} samples")
print(f"Test loader: {len(test_loader)} batches")

## 3. Load Pre-trained Models

In [ ]:
def load_model_from_checkpoint(checkpoint_path, model_class, **model_kwargs):
    """Load model from checkpoint"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    if 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    else:
        state_dict = checkpoint

    model = model_class(**model_kwargs)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()

    return model

# Model configurations
models = {}

# 1. Baseline UNet
print("Loading Baseline UNet...")
models['baseline_unet'] = load_model_from_checkpoint(
    project_root / 'checkpoints' / 'best_model.pth',
    UNet,
    n_channels=3, n_classes=3, base_channels=64
)

# 2. CLAHE (UNet trained with CLAHE preprocessing)
print("Loading CLAHE UNet...")
models['clahe'] = load_model_from_checkpoint(
    project_root / 'checkpoints_clahe' / 'best_model.pth',
    UNet,
    n_channels=3, n_classes=3, base_channels=64
)

# 3. Baseline + Std Enhancer (joint model)
print("Loading Standard Enhancer + UNet...")
enhancer = ImageEnhancer()
unet = UNet(n_channels=3, n_classes=3, base_channels=64)
joint_model = EnhancerUNetModel(enhancer, unet)

checkpoint = torch.load(project_root / 'checkpoints_enhancer' / 'phase2_best_model.pth',
                       map_location='cpu', weights_only=False)
joint_model.load_state_dict(checkpoint['model_state_dict'])
joint_model = joint_model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
joint_model.eval()
models['std_enhancer'] = joint_model

# 4. Baseline + Atrous Enhancer (joint model)
print("Loading Atrous Enhancer + UNet...")
atrous_enhancer = AtrousImageEnhancer()
unet = UNet(n_channels=3, n_classes=3, base_channels=64)
joint_model = EnhancerUNetModel(atrous_enhancer, unet)

checkpoint = torch.load(project_root / 'checkpoints_atrous_enhancer' / 'phase2_best_model.pth',
                       map_location='cpu', weights_only=False)
joint_model.load_state_dict(checkpoint['model_state_dict'])
joint_model = joint_model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
joint_model.eval()
models['atrous_enhancer'] = joint_model

# 5. ASPP Unet
print("Loading ASPP-UNet...")
models['aspp_unet'] = load_model_from_checkpoint(
    project_root / 'checkpoints_aspp_unet' / 'best_model.pth',
    ASPPUNet,
    n_channels=3, n_classes=3, base_channels=64
)

print(f"Loaded {len(models)} models successfully!")

## 4. Define Evaluation Function

In [ ]:
def count_parameters(model):
    """Count total parameters in model"""
    return sum(p.numel() for p in model.parameters())

def calculate_iou_per_class(pred, target, num_classes=3):
    """Calculate IoU for each class"""
    ious = []
    for cls in range(num_classes):
        pred_cls = (pred == cls)
        target_cls = (target == cls)

        intersection = (pred_cls & target_cls).sum().float()
        union = (pred_cls | target_cls).sum().float()

        if union > 0:
            iou = intersection / union
        else:
            iou = torch.tensor(1.0)  # Perfect score if no pixels of this class

        ious.append(iou.item())

    return ious

def evaluate_model(model, test_loader, is_enhancer_model=False):
    """Evaluate model on test dataset"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    # Loss function
    class_weights = torch.tensor([1.0, 1.0, 2.0]).to(device)  # Cup weight = 2.0
    criterion = CombinedLoss(
        ce_weight=0.5,
        dice_weight=0.5,
        class_weights=class_weights,
        device=device
    )

    model.eval()
    total_loss = 0
    total_iou_bg = 0
    total_iou_disc = 0
    total_iou_cup = 0
    num_samples = 0

    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc='Evaluating'):
            images = images.to(device)
            masks = masks.to(device)

            if is_enhancer_model:
                # For enhancer models, get both enhanced image and segmentation
                enhanced, outputs = model(images)
            else:
                # For regular models, just get segmentation
                outputs = model(images)

            # Calculate loss
            loss = criterion(outputs, masks)
            total_loss += loss.item() * images.size(0)

            # Get predictions
            pred = torch.argmax(outputs, dim=1)  # [B, H, W]

            # Calculate IoU per class for each sample
            for i in range(images.size(0)):
                ious = calculate_iou_per_class(pred[i], masks[i])
                total_iou_bg += ious[0]
                total_iou_disc += ious[1]
                total_iou_cup += ious[2]

            num_samples += images.size(0)

    # Average metrics
    avg_loss = total_loss / num_samples
    avg_iou_bg = total_iou_bg / num_samples
    avg_iou_disc = total_iou_disc / num_samples
    avg_iou_cup = total_iou_cup / num_samples
    mean_iou = (avg_iou_bg + avg_iou_disc + avg_iou_cup) / 3

    return {
        'loss': avg_loss,
        'iou_bg': avg_iou_bg,
        'iou_disc': avg_iou_disc,
        'iou_cup': avg_iou_cup,
        'mean_iou': mean_iou
    }

## 5. Evaluate Baseline UNet

In [ ]:
# Evaluate Baseline UNet
print("Evaluating Baseline UNet...")
baseline_results = evaluate_model(models['baseline_unet'], test_loader, is_enhancer_model=False)
baseline_results['parameters'] = count_parameters(models['baseline_unet'])
baseline_results['delta_miou'] = 0.0  # Reference model

print("Baseline UNet Results:")
print(f"  Loss: {baseline_results['loss']:.4f}")
print(f"  IoU BG: {baseline_results['iou_bg']:.4f}")
print(f"  IoU Disc: {baseline_results['iou_disc']:.4f}")
print(f"  IoU Cup: {baseline_results['iou_cup']:.4f}")
print(f"  mIoU: {baseline_results['mean_iou']:.4f}")
print(f"  Parameters: {baseline_results['parameters']:,}")
print(f"  Δ mIoU: {baseline_results['delta_miou']:.4f}")

## 6. Evaluate CLAHE

In [ ]:
# Evaluate CLAHE
print("Evaluating CLAHE...")
clahe_results = evaluate_model(models['clahe'], test_loader, is_enhancer_model=False)
clahe_results['parameters'] = count_parameters(models['clahe'])
clahe_results['delta_miou'] = clahe_results['mean_iou'] - baseline_results['mean_iou']

print("CLAHE Results:")
print(f"  Loss: {clahe_results['loss']:.4f}")
print(f"  IoU BG: {clahe_results['iou_bg']:.4f}")
print(f"  IoU Disc: {clahe_results['iou_disc']:.4f}")
print(f"  IoU Cup: {clahe_results['iou_cup']:.4f}")
print(f"  mIoU: {clahe_results['mean_iou']:.4f}")
print(f"  Parameters: {clahe_results['parameters']:,}")
print(f"  Δ mIoU: {clahe_results['delta_miou']:.4f}")

## 7. Evaluate Baseline + Std Enhancer

In [ ]:
# Evaluate Standard Enhancer
print("Evaluating Standard Enhancer...")
std_enhancer_results = evaluate_model(models['std_enhancer'], test_loader, is_enhancer_model=True)
std_enhancer_results['parameters'] = count_parameters(models['std_enhancer'])
std_enhancer_results['delta_miou'] = std_enhancer_results['mean_iou'] - baseline_results['mean_iou']

print("Standard Enhancer Results:")
print(f"  Loss: {std_enhancer_results['loss']:.4f}")
print(f"  IoU BG: {std_enhancer_results['iou_bg']:.4f}")
print(f"  IoU Disc: {std_enhancer_results['iou_disc']:.4f}")
print(f"  IoU Cup: {std_enhancer_results['iou_cup']:.4f}")
print(f"  mIoU: {std_enhancer_results['mean_iou']:.4f}")
print(f"  Parameters: {std_enhancer_results['parameters']:,}")
print(f"  Δ mIoU: {std_enhancer_results['delta_miou']:.4f}")

## 8. Evaluate Baseline + Atrous Enhancer

In [ ]:
# Evaluate Atrous Enhancer
print("Evaluating Atrous Enhancer...")
atrous_enhancer_results = evaluate_model(models['atrous_enhancer'], test_loader, is_enhancer_model=True)
atrous_enhancer_results['parameters'] = count_parameters(models['atrous_enhancer'])
atrous_enhancer_results['delta_miou'] = atrous_enhancer_results['mean_iou'] - baseline_results['mean_iou']

print("Atrous Enhancer Results:")
print(f"  Loss: {atrous_enhancer_results['loss']:.4f}")
print(f"  IoU BG: {atrous_enhancer_results['iou_bg']:.4f}")
print(f"  IoU Disc: {atrous_enhancer_results['iou_disc']:.4f}")
print(f"  IoU Cup: {atrous_enhancer_results['iou_cup']:.4f}")
print(f"  mIoU: {atrous_enhancer_results['mean_iou']:.4f}")
print(f"  Parameters: {atrous_enhancer_results['parameters']:,}")
print(f"  Δ mIoU: {atrous_enhancer_results['delta_miou']:.4f}")

## 9. Evaluate ASPP Unet

In [ ]:
# Evaluate ASPP-UNet
print("Evaluating ASPP-UNet...")
aspp_results = evaluate_model(models['aspp_unet'], test_loader, is_enhancer_model=False)
aspp_results['parameters'] = count_parameters(models['aspp_unet'])
aspp_results['delta_miou'] = aspp_results['mean_iou'] - baseline_results['mean_iou']

print("ASPP-UNet Results:")
print(f"  Loss: {aspp_results['loss']:.4f}")
print(f"  IoU BG: {aspp_results['iou_bg']:.4f}")
print(f"  IoU Disc: {aspp_results['iou_disc']:.4f}")
print(f"  IoU Cup: {aspp_results['iou_cup']:.4f}")
print(f"  mIoU: {aspp_results['mean_iou']:.4f}")
print(f"  Parameters: {aspp_results['parameters']:,}")
print(f"  Δ mIoU: {aspp_results['delta_miou']:.4f}")

## 10. Compile Results

In [ ]:
# Compile all results
all_results = {
    'Baseline UNet': baseline_results,
    'CLAHE': clahe_results,
    'Baseline + Std Enhancer': std_enhancer_results,
    'Baseline + Atrous Enhancer': atrous_enhancer_results,
    'ASPP Unet': aspp_results
}

# Create results DataFrame for display
results_df = pd.DataFrame.from_dict(all_results, orient='index')
results_df = results_df.round(4)

print("\n" + "="*80)
print("COMPREHENSIVE MODEL COMPARISON RESULTS")
print("="*80)
print(results_df.to_string())
print("="*80)

# Sort by mIoU for ranking
ranked_results = results_df.sort_values('mean_iou', ascending=False)
print("\nRANKED BY MEAN IOU:")
print("-" * 40)
for i, (model_name, row) in enumerate(ranked_results.iterrows(), 1):
    delta_str = f" (+{row['delta_miou']:.2%})" if row['delta_miou'] > 0 else f" ({row['delta_miou']:.2%})"
    print(f"{i}. {model_name}: {row['mean_iou']:.4f}{delta_str}")

## 11. Save Results to JSON

In [ ]:
# Save results to JSON
output_file = project_root / 'results' / 'test_comparison.json'

# Convert to JSON-serializable format
json_results = {}
for model_name, metrics in all_results.items():
    json_results[model_name] = {k: float(v) for k, v in metrics.items()}

with open(output_file, 'w') as f:
    json.dump(json_results, f, indent=2)

print(f"Results saved to: {output_file}")

# Also save the ranked results
ranked_output = {
    'ranking': [],
    'best_model': ranked_results.index[0],
    'best_miou': float(ranked_results.iloc[0]['mean_iou']),
    'baseline_miou': float(baseline_results['mean_iou'])
}

for i, (model_name, row) in enumerate(ranked_results.iterrows(), 1):
    ranked_output['ranking'].append({
        'rank': i,
        'model': model_name,
        'miou': float(row['mean_iou']),
        'delta_miou': float(row['delta_miou']),
        'parameters': int(row['parameters'])
    })

ranked_file = project_root / 'results' / 'test_ranking.json'
with open(ranked_file, 'w') as f:
    json.dump(ranked_output, f, indent=2)

print(f"Ranked results saved to: {ranked_file}")
print("\n✅ Evaluation complete! All results saved to JSON files.")

# Comprehensive Model Comparison: Optic Disc/Cup Segmentation

This notebook evaluates and compares five different approaches for optic disc and cup segmentation on the test dataset.

**Approaches:**
1. **Baseline UNet** - Standard UNet architecture
2. **CLAHE** - UNet trained with CLAHE preprocessing
3. **Baseline + Std Enhancer** - UNet with learned standard enhancer
4. **Baseline + Atrous Enhancer** - UNet with learned multi-scale enhancer
5. **ASPP Unet** - UNet with Atrous Spatial Pyramid Pooling

**Metrics:**
- Loss, IoU BG, IoU Disc, IoU Cup, mIoU, Parameters, Δ mIoU